# Stage 02 · Step 3 — Surrogate-driven τ search

Use the SSL+RUL model to predict component-level remaining life on observed telemetry, derive an expected number of preventive / corrective events for any candidate τ schedule, and let Optuna optimise τ over that surrogate without paying the simulator's full cost.

The surrogate's predictions are validated against the **real simulator** at the end: the top-K τ vectors found by the surrogate are re-simulated with `lib.env_runner.run_with_tau` and scored with `lib.objective.scalar_objective`, then compared against Stage 01's leaderboard.

If the surrogate is well-calibrated, Stage 02 returns a τ vector whose true cost is within a few percent of Stage 01's optimum — at a fraction of the wall-clock time spent inside the optimiser.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.objective import scalar_objective
from ml_models import PROJECT_ROOT
from sdg.generate import build_printer_city_map, load_configs
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from ml_models.lib.fast import SURROGATE_OPTUNA_TRIALS, PARALLEL, banner
banner()


[fast] mode=FULL · parallel=12 · trials=200/500 · K=60 · epochs=20/3 · ppo_ts=2000/20000 · seeds=(0, 1, 2)


In [ ]:
# Load the trained RUL head if it exists. The new τ search doesn't use it
# (it backs onto the real simulator), but we still load and demo it so the
# notebook prints a small sanity output proving the upstream Stage 02 work
# completed end-to-end. On a fresh checkout the .pt and .npz weights are
# gitignored, so we degrade gracefully and skip the model-based diagnostic.
_RUL_MODEL_LOADED = False
model = None
CONTEXT_LENGTH = None
channel_mean = None
channel_std = None
try:
    with (MODELS_DIR / 'ssl_config.json').open() as handle:
        saved = json.load(handle)
    patch_cfg = PatchTSTConfig(**saved['patch_cfg'])
    patch_cfg.num_targets = 6
    patch_cfg.prediction_length = 1
    patch_cfg.use_cls_token = False
    model = PatchTSTForRegression(patch_cfg).to(device)
    model.load_state_dict(torch.load(MODELS_DIR / 'rul_head_ssl.pt', map_location=device))
    model.eval()

    scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
    channel_mean = scaler['mean']; channel_std = scaler['std']
    CONTEXT_LENGTH = int(saved['train_cfg']['context_length'])
    _RUL_MODEL_LOADED = True
    print('Loaded RUL model. Context length:', CONTEXT_LENGTH)
except FileNotFoundError as exc:
    print(f'[note] RUL model artefact missing ({exc.filename}); skipping the RUL diagnostic.')
    print('       The τ search below uses the real simulator and does not need the RUL model.')

In [ ]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)

if _RUL_MODEL_LOADED:
    def predict_rul_for_window(printer_id: int, end_day: int) -> np.ndarray:
        df = filter_printers(feat_fleet, [printer_id])
        df = df.sort_values('day')
        arr = df[feature_cols].to_numpy(dtype=np.float32)
        if end_day < CONTEXT_LENGTH:
            return np.full(6, np.nan, dtype=np.float32)
        window = arr[end_day - CONTEXT_LENGTH:end_day]
        normed = (window - channel_mean) / channel_std
        x = torch.from_numpy(normed).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(past_values=x).regression_outputs.squeeze(-1).view(-1)
        return (out.cpu().numpy() * 365.0).clip(0.0, 365.0)

    sample_rul = predict_rul_for_window(VAL_PRINTERS[0], end_day=2000)
    print('Predicted RUL for printer', VAL_PRINTERS[0], 'at day 2000:', sample_rul)
else:
    print('RUL diagnostic skipped (model artefact missing).')

## τ search via the real simulator

Earlier this stage used a closed-form "surrogate" cost model
($n_{cm,i} \approx n_{cm,i}^{(0)} \cdot (\tau_i/\tau_i^{(0)})^{b_M}$,
linear PM scaling, downtime summed across components). It was
systematically biased: the SDG simulator's `apply_corrective` resets only
health, life, and the maintenance clock — *not* the global counters
`N_c`, `N_f`, `N_TC` — so post-failure cycles get shorter as a printer
ages, and the fleet baseline's `anchor_cm` balloons (C1 fires ~165k
corrective events across 85 printers over 10 yr, vs. the ~9k you'd
expect from MTTE alone). The `(τ/τ_0)^{b_M}` scaling underestimates
failures at low τ, so Optuna consistently pinned every component's τ to
the search-floor and Stage 02's validated cost ended up worse than
Stage 01.

The new objective calls `run_with_tau` (the same primitive Stage 01
uses) per Optuna trial on one stratified printer per city, and scores
with `scalar_objective`. Validation of the top-K still re-simulates on
the test printers, unchanged. The trained SSL encoder + RUL head are no
longer consumed inside the τ search — they remain upstream artifacts
that Stage 03 loads — so Stage 02-03 is functionally a Stage-01-style
search against the real simulator, but over a τ range that actually
includes the PM-effective regime (τ ≲ MTTE per component).


In [ ]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
components = components_cfg['components']
DATES = default_dates()
HORIZON_YEARS = len(DATES) / 365.25

# One stratified printer per city — same pattern as Stage 01 — so each
# Optuna trial hits every climate zone in the SDG fleet at fixed cost.
printer_city_map = build_printer_city_map(list(cities_cfg['cities']))
seen_cities: set[str] = set()
TRIAL_PRINTER_IDS: list[int] = []
for _pid, _profile in enumerate(printer_city_map):
    if _profile['name'] not in seen_cities:
        seen_cities.add(_profile['name'])
        TRIAL_PRINTER_IDS.append(_pid)
print(f'Stratified trial printers (one per city): {TRIAL_PRINTER_IDS}')

# Anchor counts (default τ_nom_d on the fleet baseline) — kept purely as
# diagnostic context for the markdown above; the new objective doesn't
# read them.
anchor_pm: dict[str, int] = {}
anchor_cm: dict[str, int] = {}
for component_id in COMPONENT_IDS:
    train_val = filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))
    anchor_pm[component_id] = int(train_val[f'maint_{component_id}'].sum())
    anchor_cm[component_id] = int(train_val[f'failure_{component_id}'].sum())
anchor_tau = {component_id: float(components[component_id]['tau_nom_d']) for component_id in COMPONENT_IDS}
print('anchor τ_nom_d (defaults):', anchor_tau)
print('anchor preventive events: ', anchor_pm)
print('anchor corrective events: ', anchor_cm)

In [ ]:
def simulator_score(tau_vector: dict[str, float]) -> dict:
    """Real-simulator cost/availability for one τ candidate.

    Replaces the earlier closed-form surrogate that pinned every component
    to the τ floor. We pay one ``run_with_tau`` rollout per trial; with
    ``n_jobs=PARALLEL`` Stage 02-03 ends up in the same wall-clock
    ballpark as Stage 01.
    """
    events = run_with_tau(
        tau_vector,
        printer_ids=TRIAL_PRINTER_IDS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    score = scalar_objective(events, components_cfg)
    return {
        'value': float(score['value']),
        'annual_cost': float(score['annual_cost']),
        'availability': float(score['availability']),
    }


# Sanity check: score the default τ_nom_d from components.yaml.
simulator_score({c: anchor_tau[c] for c in COMPONENT_IDS})

In [ ]:
# τ search ranges in DAYS. These cover the PM-effective regime for each
# component (τ ≲ MTTE so PM has a chance to fire before failure). Stage 01
# searches a much wider range that is mostly *above* MTTE and ends up in
# the corrective-dominated regime; Stage 02 here intentionally focuses on
# the regime where preventive maintenance can actually win.
TAU_RANGES = {
    'C1': (2.08, 83.33),     # 50 h .. 2000 h
    'C2': (20.83, 833.33),   # 500 h .. 20000 h
    'C3': (1.0, 20.83),      # 24 h .. 500 h
    'C4': (4.17, 83.33),     # 100 h .. 2000 h
    'C5': (20.83, 333.33),   # 500 h .. 8000 h
    'C6': (41.67, 833.33),   # 1000 h .. 20000 h
}

def trial_to_tau(trial: optuna.Trial) -> dict[str, float]:
    return {c: trial.suggest_float(f'tau_{c}', low, high, log=True) for c, (low, high) in TAU_RANGES.items()}

def surrogate_objective(trial: optuna.Trial) -> float:
    """Optuna objective. Name kept as 'surrogate' for compatibility with
    downstream cells; it now backs onto the real simulator."""
    tau_vector = trial_to_tau(trial)
    score = simulator_score(tau_vector)
    for key in ('annual_cost', 'availability'):
        trial.set_user_attr(key, float(score[key]))
    return float(score['value'])

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=99),
)
study.optimize(surrogate_objective, n_trials=SURROGATE_OPTUNA_TRIALS, n_jobs=PARALLEL, show_progress_bar=True)
study_df = study.trials_dataframe().sort_values('value').reset_index(drop=True)
study_df.head(5)

In [7]:
TOP_K = 5
real_results = []
for _, row in study_df.head(TOP_K).iterrows():
    tau_vector = {c: float(row[f'params_tau_{c}']) for c in COMPONENT_IDS}
    events = run_with_tau(
        tau_vector,
        printer_ids=TEST_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    real_score = scalar_objective(events, components_cfg)
    real_results.append({
        'trial': int(row['number']),
        'surrogate_value': float(row['value']),
        'surrogate_annual_cost': float(row['user_attrs_annual_cost']),
        'real_value': float(real_score['value']),
        'real_annual_cost': float(real_score['annual_cost']),
        'real_availability': float(real_score['availability']),
        **{f'tau_{c}': tau_vector[c] for c in COMPONENT_IDS},
    })
real_df = pd.DataFrame(real_results).sort_values('real_value').reset_index(drop=True)
real_df

,trial,surrogate_value,surrogate_annual_cost,real_value,real_annual_cost,real_availability,tau_C1,tau_C2,tau_C3,tau_C4,tau_C5,tau_C6
0,403,135043.851127,135043.851127,1.050000e+10,4.044317e+06,0.0,2.080770,30.108913,19.243376,82.963079,23.522210,775.703691
1,442,135112.348842,135112.348842,1.050000e+10,4.043827e+06,0.0,2.081791,24.882185,19.680343,82.282606,22.443172,714.215837
2,371,135173.895363,135173.895363,1.050000e+10,4.032210e+06,0.0,2.080086,27.422296,20.811095,77.201898,28.182554,677.862020
3,482,135184.960188,135184.960188,1.050000e+10,4.041440e+06,0.0,2.083276,25.900568,19.678728,76.270575,26.297860,748.284277
4,418,135186.934920,135186.934920,1.050000e+10,4.050124e+06,0.0,2.084415,31.507133,18.541069,79.518243,23.478456,775.768345


In [8]:
winner = real_df.iloc[0]
best_tau = {c: float(winner[f'tau_{c}']) for c in COMPONENT_IDS}
payload = {
    'tau_nom_h': best_tau,
    'validated_on': 'test printers (id 85..99)',
    'surrogate_annual_cost': float(winner['surrogate_annual_cost']),
    'real_annual_cost_eur_per_printer_year': float(winner['real_annual_cost']),
    'real_availability': float(winner['real_availability']),
}
with (RESULTS_DIR / 'best_tau_surrogate.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print(yaml.safe_dump(payload, sort_keys=False))

tau_nom_h:
  C1: 2.080770216460065
  C2: 30.108912662777
  C3: 19.243375568601316
  C4: 82.96307868162094
  C5: 23.52220990967777
  C6: 775.7036912161778
validated_on: test printers (id 85..99)
surrogate_annual_cost: 135043.85112705317
real_annual_cost_eur_per_printer_year: 4044316.6965507804
real_availability: 0.0

